# FINAL FREEZE – Self-Supervised Representation Learning für Phishing-Webseiten

Dieses Notebook implementiert den eingefrorenen Versuchsaufbau der Bachelorarbeit:

- **R0 BASE:** RoBERTa-base ohne zusätzliches domänenspezifisches SSL
- **R1 DAPT:** vorhandene DAPT-40k-Encoder (MLM) werden wiederverwendet
- **R2 CONTRASTIVE:** unüberwachtes, SimCSE-artiges Contrastive Learning auf demselben 40k-Pool
- **Downstream:** Logistische Regression, XGBoost, kleines MLP
- **Labelbudgets:** 10 %, 25 %, 100 % der 4.000 gelabelten Trainingsseiten
- **Evaluation:** IID, temporal, domain-disjoint, template-disjoint, domain+template-disjoint
- **Auswertung:** absolute Leistung, SSL-Effekt gegenüber BASE, Degradation unter Shift

## Methodischer Freeze

1. Test-/OOD-Daten werden **nicht** für SSL-Training oder Hyperparameterwahl verwendet.
2. BASE, DAPT und CONTRASTIVE verwenden **dieselbe Tokenisierung und denselben Input `text`**.
3. Contrastive Learning erhält **keine Labels** und **keine zusätzliche Modalität**. Positive Paare entstehen wie bei der unüberwachten SimCSE-Idee aus zwei Dropout-Views derselben Eingabe.
4. Hyperparameter der drei Downstream-Klassifikatoren werden ausschließlich auf dem gelabelten Trainingssplit per innerer Cross-Validation abgestimmt und danach eingefroren.
5. Dieselben verschachtelten Label-Subsets werden innerhalb eines Seeds für alle Repräsentationen und Klassifikatoren verwendet.
6. `Average Precision` ist die zentrale schwellenwertfreie Metrik. Zusätzlich werden Precision, Recall, F1 und empirische FPR an einem **nur auf einem separaten Kalibrierungsanteil** bestimmten Schwellenwert ausgewiesen.

**Literaturanker der SSL-Methoden:** Gururangan et al. (2020), DAPT; Gao et al. (2021), SimCSE.

> **Notebook-Version v2:** Loader-Hotfix für historische `dapt40k_bundle.pkl`-Bundles mit dem Key `pretrain_large_df`.

> **Notebook-Version v3:** OOD-Szenarien werden schemaunabhängig aus DAPT-40k + Train + Validation rekonstruiert; Near-Duplicate-Flags werden aus dem 40k-DAPT-Holdout übernommen und gegen die historischen Szenariogrößen regressionsgeprüft.


In [ ]:
# ============================================================
# 00 – Imports und globaler Freeze
# ============================================================

import os, re, gc, json, time, math, pickle, random, hashlib, shutil, warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_score, recall_score,
    f1_score, confusion_matrix
)

from scipy import stats
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/phreshphish_final_freeze_output')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 52, 62, 72, 82]
REPRESENTATIONS = ['BASE', 'DAPT', 'CONTRASTIVE']
CLASSIFIERS = ['LOGREG', 'XGBOOST', 'MLP']
LABEL_BUDGETS = [0.10, 0.25, 1.00]
TARGET_FPR = 0.005
MAX_LENGTH = 256

# 30 % des bisherigen Validation-Splits dienen ausschließlich zur Schwellenwert-Kalibrierung.
# Die übrigen 70 % bilden die IID-Referenz. Der externe Holdout bleibt davon vollständig unberührt.
CALIBRATION_FRACTION = 0.30
CALIBRATION_SPLIT_SEED = 20260808

# Contrastive Freeze (SimCSE-artig, unlabeled, gleiche Inputs wie BASE/DAPT)
CONTRASTIVE_EPOCHS = 1
CONTRASTIVE_BATCH_SIZE = 16
CONTRASTIVE_LR = 1e-5
CONTRASTIVE_WEIGHT_DECAY = 0.01
CONTRASTIVE_TEMPERATURE = 0.05
CONTRASTIVE_CHECKPOINT_EVERY = 500

# Embedding-Extraktion
EMBED_BATCH_SIZE = 64
TOKENIZE_BATCH_SIZE = 512
NUM_WORKERS = 2

# Tuning: bewusst klein und vorab fixiert
CV_FOLDS = 3
TUNING_SEED = 20260808

RUN_CONTRASTIVE = True
RUN_EMBEDDINGS = True
RUN_DOWNSTREAM = True
RUN_PLOTS = True
RUN_ACTIVE_LEARNING = False  # explizit außerhalb des Kern-Freeze

print({
    'torch': torch.__version__,
    'cuda': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'output': str(OUTPUT_ROOT),
})

## 01 – Artefakt-Audit

Das Notebook sucht rekursiv unter `/kaggle/input`. Dadurch dürfen die Kaggle-Dataset-Namen frei gewählt werden. Erforderlich sind:

- `split_roles_and_holdout_cache_v2_ram_safe.pkl`
- `dapt40k_bundle.pkl`
- lokales `roberta-base`-Modell
- fünf vorhandene DAPT-40k-Encoder für Seeds 42, 52, 62, 72, 82

Phase-5D-/Active-Learning-Ergebnisse werden für den neuen Freeze **nicht benötigt**.

In [ ]:
# ============================================================
# 01 – Dateisuche und Artefakt-Audit
# ============================================================

def all_paths_named(name: str) -> List[Path]:
    return sorted(INPUT_ROOT.rglob(name))


def first_path_named(name: str, required=True) -> Optional[Path]:
    matches = all_paths_named(name)
    if not matches:
        if required:
            raise FileNotFoundError(f'{name} wurde unter {INPUT_ROOT} nicht gefunden.')
        return None
    return matches[0]


def is_hf_model_dir(path: Path) -> bool:
    if not path.is_dir() or not (path / 'config.json').exists():
        return False
    return any((path / f).exists() for f in ['model.safetensors', 'pytorch_model.bin'])


def find_base_roberta() -> Path:
    candidates=[]
    for cfg in INPUT_ROOT.rglob('config.json'):
        d=cfg.parent
        low=str(d).lower()
        if is_hf_model_dir(d) and 'roberta' in low and 'dapt_encoder' not in low and '/seed_' not in low.replace('\\','/'):
            # Bevorzuge explizit roberta-base benannte Verzeichnisse.
            score = 0 if 'roberta-base' in d.name.lower() or 'roberta_base' in d.name.lower() else 1
            candidates.append((score, len(str(d)), d))
    if not candidates:
        raise FileNotFoundError('Kein lokales RoBERTa-base-Modell unter /kaggle/input gefunden.')
    return sorted(candidates)[0][2]


def find_dapt_encoder(seed: int) -> Path:
    candidates=[]
    for cfg in INPUT_ROOT.rglob('config.json'):
        d=cfg.parent
        low=str(d).lower().replace('\\','/')
        if not is_hf_model_dir(d):
            continue
        if f'seed_{seed}' not in low:
            continue
        # Strikt die 40k-DAPT-RoBERTa-L256-Linie bevorzugen.
        score = 0
        if 'dapt_encoders/40k/roberta_l256' not in low:
            score += 10
        if 'dapt' not in low:
            score += 100
        candidates.append((score, len(low), d))
    if not candidates:
        raise FileNotFoundError(f'DAPT-40k-Encoder für Seed {seed} nicht gefunden.')
    best=sorted(candidates)[0]
    if best[0] >= 100:
        raise FileNotFoundError(f'Kein eindeutig als DAPT erkennbarer Encoder für Seed {seed}.')
    return best[2]

SPLIT_CACHE = first_path_named('split_roles_and_holdout_cache_v2_ram_safe.pkl')
DAPT_BUNDLE = first_path_named('dapt40k_bundle.pkl')
BASE_MODEL_DIR = find_base_roberta()
DAPT_DIRS = {seed: find_dapt_encoder(seed) for seed in SEEDS}

AUDIT = {
    'split_cache': str(SPLIT_CACHE),
    'dapt_bundle': str(DAPT_BUNDLE),
    'base_model': str(BASE_MODEL_DIR),
    'dapt_encoders': {str(k): str(v) for k,v in DAPT_DIRS.items()},
}

print(json.dumps(AUDIT, indent=2))
(OUTPUT_ROOT / 'artifact_audit.json').write_text(json.dumps(AUDIT, indent=2), encoding='utf-8')

## 02 – Split-Cache und 40k-SSL-Pool laden

Der Loader akzeptiert mehrere historische Key-Namen, bricht aber ab, wenn der bereits verwendete Input `text` nicht vorhanden ist. Es wird **keine neue URL/HTML-Serialisierung erfunden**, weil dies die Vergleichbarkeit zu den vorhandenen DAPT-Encodern verändern würde.

In [ ]:
# ============================================================
# 02 – Robuster Cache-Loader
# ============================================================

def load_pickle(path: Path):
    with path.open('rb') as f:
        return pickle.load(f)


def nested_items(obj, prefix='root'):
    if isinstance(obj, dict):
        for k,v in obj.items():
            p=f'{prefix}.{k}'
            yield p,k,v
            yield from nested_items(v,p)


def find_nested_value(obj, aliases: Iterable[str], expected_type=None):
    aliases={a.lower() for a in aliases}
    if isinstance(obj, dict):
        for _,k,v in nested_items(obj):
            if str(k).lower() in aliases and (expected_type is None or isinstance(v, expected_type)):
                return v
    return None


def as_frame(obj, name):
    if isinstance(obj, pd.DataFrame):
        return obj.reset_index(drop=True)
    if isinstance(obj, list) and (len(obj)==0 or isinstance(obj[0], dict)):
        return pd.DataFrame(obj).reset_index(drop=True)
    raise TypeError(f'{name} ist kein DataFrame/Record-Array, sondern {type(obj)}')

split_payload = load_pickle(SPLIT_CACHE)
dapt_payload = load_pickle(DAPT_BUNDLE)

train_obj = find_nested_value(split_payload, ['train_df','train','supervised_train','downstream_train'])
val_obj = find_nested_value(split_payload, ['val_df','validation_df','validation','val'])
holdout_obj = find_nested_value(split_payload, ['final_holdout_clean','final_holdout','holdout_df','holdout'])

# Historische Split-Caches enthalten nicht zwingend materialisierte Szenarien oder
# `seen_domains`. Die finalen OOD-Szenarien werden deshalb in 02b explizit aus
# PRETRAIN(40k)+TRAIN+VALIDATION rekonstruiert.
scenario_obj = find_nested_value(split_payload, ['scenarios','scenario_frames','scenario_indices'])

if train_obj is None or val_obj is None or holdout_obj is None:
    top_keys = list(split_payload.keys()) if isinstance(split_payload, dict) else [type(split_payload).__name__]
    raise KeyError(f'Train/Validation/Holdout konnten nicht eindeutig aus dem Split-Cache gelesen werden. Top-Level: {top_keys}')

train_df = as_frame(train_obj, 'train')
val_df = as_frame(val_obj, 'validation')
holdout_df = as_frame(holdout_obj, 'holdout')

if isinstance(dapt_payload, pd.DataFrame):
    pretrain_df = dapt_payload.reset_index(drop=True)
else:
    # Historische Bundle-Versionen verwendeten unterschiedliche Namen.
    # Das aktuell vorhandene ENDGAME-Bundle speichert den 40k-Pool als
    # `pretrain_large_df`; ältere Varianten nutzten u. a. `frame`.
    dapt_obj = find_nested_value(
        dapt_payload,
        [
            'pretrain_large_df',
            'frame',
            'pretrain_df',
            'pretrain',
            'dapt_40k',
            'dapt_unlabeled',
            'data',
        ],
    )
    if dapt_obj is None:
        top_keys = list(dapt_payload.keys()) if isinstance(dapt_payload, dict) else [type(dapt_payload).__name__]
        raise KeyError(
            '40k-Frame nicht in dapt40k_bundle.pkl gefunden. '
            f'Erkannte Top-Level-Keys: {top_keys}'
        )
    pretrain_df = as_frame(dapt_obj, 'dapt40k')

# Das DAPT-40k-Bundle enthält in der vorhandenen ENDGAME-Version zusätzlich
# `final_holdout_clean`. Dieser Frame wurde nach der 40k-Leakage-/Overlap-
# Prüfung gespeichert und ist daher die bevorzugte Quelle für Near-Duplicate-
# Flags. Der Split-Cache bleibt die kanonische Quelle für die Holdout-Reihenfolge.
dapt_holdout_obj = None
if isinstance(dapt_payload, dict):
    dapt_holdout_obj = dapt_payload.get('final_holdout_clean')

if isinstance(dapt_holdout_obj, pd.DataFrame):
    dapt_holdout_flags_df = dapt_holdout_obj.reset_index(drop=True)
else:
    dapt_holdout_flags_df = None

print({
    'dapt_bundle_top_keys': list(dapt_payload.keys()) if isinstance(dapt_payload, dict) else type(dapt_payload).__name__,
    'dapt_frame_rows': len(pretrain_df),
    'dapt_frame_columns': pretrain_df.columns.tolist(),
    'dapt_holdout_rows': None if dapt_holdout_flags_df is None else len(dapt_holdout_flags_df),
    'dapt_holdout_columns': None if dapt_holdout_flags_df is None else dapt_holdout_flags_df.columns.tolist(),
})

for name, frame, need_label in [
    ('train',train_df,True), ('validation',val_df,True), ('holdout',holdout_df,True), ('dapt40k',pretrain_df,False)
]:
    if 'text' not in frame.columns:
        raise KeyError(f'{name}: Spalte `text` fehlt. Keine automatische Neu-Serialisierung, um DAPT-Kompatibilität zu erhalten. Vorhanden: {frame.columns.tolist()}')
    if need_label and 'label' not in frame.columns:
        raise KeyError(f'{name}: Spalte `label` fehlt.')

for frame in [train_df,val_df,holdout_df]:
    frame['label'] = frame['label'].astype(int)

if len(pretrain_df) != 40000:
    raise RuntimeError(f'Erwartet werden 40.000 DAPT/Contrastive-Seiten, gefunden: {len(pretrain_df)}')

holdout_df = holdout_df.copy()

# 40k-aware Zusatzflags per SHA256 aus dem DAPT-Bundle übernehmen.
# Es werden KEINE Labels oder Testinformationen in Training/Tuning eingespeist;
# diese Flags dienen ausschließlich der vorab definierten Szenariokonstruktion.
if dapt_holdout_flags_df is not None:
    if 'sha256' not in holdout_df.columns or 'sha256' not in dapt_holdout_flags_df.columns:
        raise KeyError('Für den sicheren Holdout-Flag-Abgleich fehlt `sha256`.')
    if holdout_df['sha256'].astype(str).duplicated().any():
        raise RuntimeError('Holdout-SHA256 ist nicht eindeutig.')
    if dapt_holdout_flags_df['sha256'].astype(str).duplicated().any():
        raise RuntimeError('DAPT-Bundle-Holdout-SHA256 ist nicht eindeutig.')

    aux_cols = [
        c for c in [
            'near_duplicate_to_development',
            'min_simhash_distance_to_development',
            'template_seen_in_development',
        ]
        if c in dapt_holdout_flags_df.columns
    ]
    if aux_cols:
        aux = dapt_holdout_flags_df[['sha256'] + aux_cols].copy()
        aux['sha256'] = aux['sha256'].astype(str)
        base_sha = holdout_df['sha256'].astype(str)
        aux = aux.set_index('sha256')
        missing_sha = ~base_sha.isin(aux.index)
        if missing_sha.any():
            raise RuntimeError(
                f'DAPT-Bundle-Holdout passt nicht zum Split-Holdout: '
                f'{int(missing_sha.sum())} SHA256 fehlen.'
            )
        for c in aux_cols:
            # Explizit überschreiben: die Bundle-Flags stammen aus der
            # 40k-Development-Konfiguration und sind für diesen Freeze maßgeblich.
            holdout_df[c] = base_sha.map(aux[c]).to_numpy()

holdout_df['_holdout_row'] = np.arange(len(holdout_df), dtype=np.int32)

print({
    'train': len(train_df),
    'validation_original': len(val_df),
    'holdout': len(holdout_df),
    'dapt_unlabeled': len(pretrain_df),
    'train_labels': train_df.label.value_counts().to_dict(),
    'validation_labels': val_df.label.value_counts().to_dict(),
    'holdout_labels': holdout_df.label.value_counts().to_dict(),
})

In [ ]:
# ============================================================
# 02b – IID-Kalibrierung/Test und reproduzierbare OOD-Szenarien
# ============================================================

cal_idx, iid_idx = train_test_split(
    np.arange(len(val_df)),
    test_size=1.0-CALIBRATION_FRACTION,
    random_state=CALIBRATION_SPLIT_SEED,
    stratify=val_df['label'].to_numpy(),
)
calibration_df = val_df.iloc[np.sort(cal_idx)].reset_index(drop=True)
iid_df = val_df.iloc[np.sort(iid_idx)].reset_index(drop=True)


# ------------------------------------------------------------
# A) Harte Schema-Prüfung
# ------------------------------------------------------------
required_identity = ['sha256', 'domain', 'template_hash']
for name, frame in [
    ('dapt40k', pretrain_df),
    ('train', train_df),
    ('validation', val_df),
    ('holdout', holdout_df),
]:
    missing = [c for c in required_identity if c not in frame.columns]
    if missing:
        raise KeyError(
            f'{name}: Für die Freeze-Szenarien fehlen Spalten {missing}. '
            f'Vorhanden: {frame.columns.tolist()}'
        )

if holdout_df['sha256'].astype(str).duplicated().any():
    raise RuntimeError('Holdout-SHA256 ist nicht eindeutig.')


# ------------------------------------------------------------
# B) "Gesehen" bedeutet: während Development/SSL verfügbar.
#    Das entspricht der historischen 40k-Szenariodefinition:
#    DAPT-40k + Downstream-Train + ursprüngliche Validation.
# ------------------------------------------------------------
development_domains = set()
development_templates = set()

for frame in [pretrain_df, train_df, val_df]:
    development_domains.update(
        frame['domain'].fillna('').astype(str).tolist()
    )
    development_templates.update(
        frame['template_hash'].fillna('').astype(str).tolist()
    )

# Leere Identifikatoren sollen keine echte Domäne/Template-Familie darstellen.
development_domains.discard('')
development_templates.discard('')

holdout_domain = holdout_df['domain'].fillna('').astype(str)
holdout_template = holdout_df['template_hash'].fillna('').astype(str)

domain_seen_40k = holdout_domain.isin(development_domains).to_numpy()
template_seen_40k = holdout_template.isin(development_templates).to_numpy()

# Explizit neu setzen, damit die Szenarien nicht von historisch anders
# benannten/älteren Cache-Flags abhängen.
holdout_df['domain_seen_in_development_freeze'] = domain_seen_40k
holdout_df['template_seen_in_development_freeze'] = template_seen_40k


# ------------------------------------------------------------
# C) Near-Duplicate-Flag für strukturelles OOD
# ------------------------------------------------------------
# Die historische Stressdefinition schließt bei Template-OOD zusätzlich
# Near-Duplicates zum Development aus. Bevorzugt wird der 40k-aware Flag
# aus `dapt40k_bundle.pkl -> final_holdout_clean`.
if 'near_duplicate_to_development' not in holdout_df.columns:
    # Fallback: gezielt nach einem bereits materialisierten 40k-Holdout suchen.
    fallback_candidates = (
        all_paths_named('final_holdout_with_40k_overlap_flags.pkl')
        + all_paths_named('final_holdout_with_overlap_flags.pkl')
    )
    loaded = False
    for candidate in fallback_candidates:
        try:
            aux = load_pickle(candidate)
            if not isinstance(aux, pd.DataFrame):
                continue
            if not {'sha256','near_duplicate_to_development'}.issubset(aux.columns):
                continue
            if aux['sha256'].astype(str).duplicated().any():
                continue
            lookup = aux.set_index(aux['sha256'].astype(str))['near_duplicate_to_development']
            mapped = holdout_df['sha256'].astype(str).map(lookup)
            if mapped.isna().any():
                continue
            holdout_df['near_duplicate_to_development'] = mapped.astype(bool).to_numpy()
            print({'near_duplicate_flags_source': str(candidate)})
            loaded = True
            break
        except Exception as exc:
            print({'near_duplicate_fallback_skipped': str(candidate), 'reason': str(exc)})
    if not loaded:
        raise RuntimeError(
            'Near-Duplicate-Flags fehlen. Für TEMPLATE_OOD wird nicht stillschweigend '
            'auf eine schwächere Definition zurückgefallen.'
        )
else:
    print({'near_duplicate_flags_source': 'dapt40k_bundle/final_holdout_clean'})


near_dup = holdout_df['near_duplicate_to_development'].fillna(False).astype(bool).to_numpy()


# ------------------------------------------------------------
# D) Deterministisches Balancing analog zur bisherigen Stresslogik
# ------------------------------------------------------------
def balanced_available_indices(frame: pd.DataFrame, mask, seed: int):
    sub = frame.loc[np.asarray(mask)].copy()
    n0 = int((sub['label'] == 0).sum())
    n1 = int((sub['label'] == 1).sum())
    n_each = min(n0, n1)
    if n_each == 0:
        raise RuntimeError(
            f'Szenario enthält nicht beide Klassen: label0={n0}, label1={n1}'
        )

    # Seed-Konvention aus der bisherigen Stresspipeline.
    part0 = sub[sub['label'].eq(0)].sample(n=n_each, random_state=seed)
    part1 = sub[sub['label'].eq(1)].sample(n=n_each, random_state=seed + 1)
    out = pd.concat([part0, part1], axis=0)
    # Reihenfolge ist für die Evaluation irrelevant; über den Holdout-Index
    # wird dennoch eine kanonische Reihenfolge hergestellt.
    return np.sort(out.index.to_numpy(dtype=np.int32))


SCENARIO_INDICES = {}

# Temporal: der finale Holdout ist bereits 4k/4k balanciert.
if int((holdout_df.label == 0).sum()) != int((holdout_df.label == 1).sum()):
    raise RuntimeError('Finaler Temporal-Holdout ist unerwartet nicht balanciert.')
SCENARIO_INDICES['final_temporal_balanced'] = np.arange(
    len(holdout_df), dtype=np.int32
)

domain_new_mask = (~domain_seen_40k) & holdout_domain.ne('').to_numpy()
template_ood_mask = (
    (~template_seen_40k)
    & (~near_dup)
    & holdout_template.ne('').to_numpy()
)
domain_template_ood_mask = domain_new_mask & template_ood_mask

SCENARIO_INDICES['new_domain_balanced'] = balanced_available_indices(
    holdout_df, domain_new_mask, 108
)
SCENARIO_INDICES['template_disjoint_balanced'] = balanced_available_indices(
    holdout_df, template_ood_mask, 109
)
SCENARIO_INDICES['domain_template_disjoint_balanced'] = balanced_available_indices(
    holdout_df, domain_template_ood_mask, 110
)


# ------------------------------------------------------------
# E) Regressions-Audit gegen die bereits materialisierten Stressszenarien
# ------------------------------------------------------------
EXPECTED_HISTORICAL_SCENARIO_ROWS = {
    'final_temporal_balanced': 8000,
    'new_domain_balanced': 3984,
    'template_disjoint_balanced': 6832,
    'domain_template_disjoint_balanced': 3822,
}

scenario_regression = {}
for name, expected_n in EXPECTED_HISTORICAL_SCENARIO_ROWS.items():
    idx = SCENARIO_INDICES[name]
    actual_n = int(len(idx))
    labels = holdout_df.iloc[idx]['label']
    row = {
        'expected_n': int(expected_n),
        'actual_n': actual_n,
        'benign': int((labels == 0).sum()),
        'phish': int((labels == 1).sum()),
        'matches_expected_n': bool(actual_n == expected_n),
    }
    scenario_regression[name] = row

print('Scenario regression audit:')
print(json.dumps(scenario_regression, indent=2))

mismatches = {
    name: row
    for name, row in scenario_regression.items()
    if not row['matches_expected_n']
}
if mismatches:
    raise RuntimeError(
        'Freeze-Szenarien weichen von den bereits dokumentierten historischen '
        f'Größen ab. Kein Training gestartet. Abweichungen: {mismatches}'
    )


# ------------------------------------------------------------
# F) Finale Evaluation-Mappings
# ------------------------------------------------------------
SCENARIOS = {
    'IID': ('iid', np.arange(len(iid_df), dtype=np.int32)),
    'TEMPORAL': ('holdout', SCENARIO_INDICES['final_temporal_balanced']),
    'DOMAIN_OOD': ('holdout', SCENARIO_INDICES['new_domain_balanced']),
    'TEMPLATE_OOD': ('holdout', SCENARIO_INDICES['template_disjoint_balanced']),
    'DOMAIN_TEMPLATE_OOD': (
        'holdout',
        SCENARIO_INDICES['domain_template_disjoint_balanced'],
    ),
}

scenario_summary = []
for name, (source, idx) in SCENARIOS.items():
    frame = iid_df.iloc[idx] if source == 'iid' else holdout_df.iloc[idx]
    scenario_summary.append({
        'scenario': name,
        'source': source,
        'n': int(len(frame)),
        'benign': int((frame.label == 0).sum()),
        'phish': int((frame.label == 1).sum()),
    })

scenario_summary_df = pd.DataFrame(scenario_summary)
print(scenario_summary_df.to_string(index=False))
print({
    'calibration_n': len(calibration_df),
    'iid_n': len(iid_df),
    'development_unique_domains': len(development_domains),
    'development_unique_templates': len(development_templates),
    'holdout_near_duplicates': int(near_dup.sum()),
})

scenario_summary_df.to_csv(
    OUTPUT_ROOT / 'scenario_summary.csv',
    index=False,
)
(OUTPUT_ROOT / 'scenario_regression_audit.json').write_text(
    json.dumps(scenario_regression, indent=2),
    encoding='utf-8',
)


## 03 – Einheitliche Token-Caches

Alle drei Repräsentationsvarianten erhalten exakt dieselben Token-IDs. Dies verhindert, dass ein vermeintlicher SSL-Effekt aus unterschiedlicher Vorverarbeitung entsteht.

In [ ]:
# ============================================================
# 03 – Tokenizer und Token-Caches
# ============================================================

TOKEN_ROOT=OUTPUT_ROOT/'tokens'
TOKEN_ROOT.mkdir(exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR, local_files_only=True, use_fast=True)


def tokenize_frame(frame: pd.DataFrame, name: str, include_labels: bool):
    root=TOKEN_ROOT/name
    root.mkdir(parents=True,exist_ok=True)
    ids_path=root/'input_ids.npy'
    mask_path=root/'attention_mask.npy'
    labels_path=root/'labels.npy'
    meta_path=root/'meta.json'
    n=len(frame)

    if ids_path.exists() and mask_path.exists() and meta_path.exists():
        meta=json.loads(meta_path.read_text())
        if meta.get('n')==n and meta.get('max_length')==MAX_LENGTH:
            return {
                'input_ids':np.load(ids_path,mmap_mode='r'),
                'attention_mask':np.load(mask_path,mmap_mode='r'),
                'labels':np.load(labels_path,mmap_mode='r') if include_labels and labels_path.exists() else None,
            }

    input_ids=np.lib.format.open_memmap(ids_path,mode='w+',dtype=np.int32,shape=(n,MAX_LENGTH))
    attention=np.lib.format.open_memmap(mask_path,mode='w+',dtype=np.uint8,shape=(n,MAX_LENGTH))

    texts=frame['text'].fillna('').astype(str).tolist()
    for start in range(0,n,TOKENIZE_BATCH_SIZE):
        end=min(start+TOKENIZE_BATCH_SIZE,n)
        enc=tokenizer(
            texts[start:end], padding='max_length', truncation=True,
            max_length=MAX_LENGTH, return_tensors='np'
        )
        input_ids[start:end]=enc['input_ids'].astype(np.int32)
        attention[start:end]=enc['attention_mask'].astype(np.uint8)
        if end % 5000 == 0 or end==n:
            print({'tokenizing':name,'done':end,'total':n})
    input_ids.flush(); attention.flush()

    if include_labels:
        labels=np.lib.format.open_memmap(labels_path,mode='w+',dtype=np.int8,shape=(n,))
        labels[:]=frame['label'].to_numpy(dtype=np.int8)
        labels.flush(); del labels

    meta_path.write_text(json.dumps({'n':n,'max_length':MAX_LENGTH},indent=2))
    del input_ids,attention,texts
    gc.collect()
    return {
        'input_ids':np.load(ids_path,mmap_mode='r'),
        'attention_mask':np.load(mask_path,mmap_mode='r'),
        'labels':np.load(labels_path,mmap_mode='r') if include_labels else None,
    }

TOKENS={
    'pretrain':tokenize_frame(pretrain_df,'pretrain_40k',False),
    'train':tokenize_frame(train_df,'train_4k',True),
    'calibration':tokenize_frame(calibration_df,'calibration',True),
    'iid':tokenize_frame(iid_df,'iid_test',True),
    'holdout':tokenize_frame(holdout_df,'holdout_8k',True),
}

print({k:{kk:(None if vv is None else list(vv.shape)) for kk,vv in v.items()} for k,v in TOKENS.items()})

## 04 – Contrastive SSL

Die Implementierung folgt dem **unüberwachten SimCSE-Prinzip**: dieselbe tokenisierte Webseite wird zweimal im Trainingsmodus durch denselben Encoder geschickt. Standard-Dropout erzeugt die beiden stochastischen Views. Andere Beispiele desselben Batches dienen als In-Batch-Negatives. Labels werden nicht geladen.

Der Encoder startet für jeden Seed erneut vom identischen `roberta-base`-Checkpoint. Das Ergebnis wird pro Seed gespeichert und bei einem erneuten Kaggle-Lauf automatisch wiederverwendet, falls ein vorheriger Output als Input eingebunden wurde.

In [ ]:
# ============================================================
# 04 – SimCSE-artiges Contrastive Learning
# ============================================================

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CONTR_ROOT=OUTPUT_ROOT/'contrastive_encoders'
CONTR_ROOT.mkdir(exist_ok=True)

class TokenArrayDataset(Dataset):
    def __init__(self, arrays, indices=None):
        self.ids=arrays['input_ids']; self.mask=arrays['attention_mask']
        self.indices=np.arange(len(self.ids)) if indices is None else np.asarray(indices)
    def __len__(self): return len(self.indices)
    def __getitem__(self,i):
        j=int(self.indices[i])
        return torch.from_numpy(np.asarray(self.ids[j],dtype=np.int64)), torch.from_numpy(np.asarray(self.mask[j],dtype=np.int64))


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def contrastive_loss(z1,z2,temp):
    z1=F.normalize(z1,p=2,dim=1); z2=F.normalize(z2,p=2,dim=1)
    logits=torch.matmul(z1,z2.T)/temp
    labels=torch.arange(z1.size(0),device=z1.device)
    return (F.cross_entropy(logits,labels)+F.cross_entropy(logits.T,labels))/2


def find_existing_contrastive(seed):
    # Aktueller Output zuerst
    local=CONTR_ROOT/f'seed_{seed}'
    if is_hf_model_dir(local): return local
    # Danach frühere Kaggle Outputs/Datasets
    candidates=[]
    for cfg in INPUT_ROOT.rglob('config.json'):
        d=cfg.parent; low=str(d).lower().replace('\\','/')
        if is_hf_model_dir(d) and f'seed_{seed}' in low and 'contrastive' in low:
            candidates.append(d)
    return sorted(candidates,key=lambda p:len(str(p)))[0] if candidates else None


def find_resume(seed):
    matches=all_paths_named(f'contrastive_resume_seed_{seed}.pt')
    return matches[0] if matches else None


def train_contrastive(seed):
    existing=find_existing_contrastive(seed)
    if existing is not None:
        print({'contrastive_seed':seed,'status':'REUSE','path':str(existing)})
        return existing, []
    if not RUN_CONTRASTIVE:
        raise FileNotFoundError(f'Contrastive Seed {seed} fehlt und RUN_CONTRASTIVE=False.')

    set_seed(seed)
    model=AutoModel.from_pretrained(BASE_MODEL_DIR,local_files_only=True)
    model.to(DEVICE); model.train()

    ds=TokenArrayDataset(TOKENS['pretrain'])
    g=torch.Generator(); g.manual_seed(seed)
    loader=DataLoader(ds,batch_size=CONTRASTIVE_BATCH_SIZE,shuffle=True,generator=g,
                      num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available(),drop_last=True)

    optimizer=torch.optim.AdamW(model.parameters(),lr=CONTRASTIVE_LR,weight_decay=CONTRASTIVE_WEIGHT_DECAY)
    scaler=torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
    start_epoch=0; global_step=0
    resume=find_resume(seed)
    if resume is not None:
        state=torch.load(resume,map_location='cpu')
        model.load_state_dict(state['model']); optimizer.load_state_dict(state['optimizer'])
        try: scaler.load_state_dict(state['scaler'])
        except Exception: pass
        start_epoch=int(state.get('epoch',0)); global_step=int(state.get('global_step',0))
        print({'contrastive_seed':seed,'resume':str(resume),'global_step':global_step})

    history=[]; t0=time.perf_counter()
    for epoch in range(start_epoch,CONTRASTIVE_EPOCHS):
        running=0.0; seen=0
        for step,(ids,mask) in enumerate(loader):
            ids=ids.to(DEVICE,non_blocking=True); mask=mask.to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda',enabled=torch.cuda.is_available()):
                # Zwei Forward-Pässe derselben Eingabe; Dropout erzeugt die Views.
                z1=model(input_ids=ids,attention_mask=mask).last_hidden_state[:,0,:]
                z2=model(input_ids=ids,attention_mask=mask).last_hidden_state[:,0,:]
                loss=contrastive_loss(z1,z2,CONTRASTIVE_TEMPERATURE)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(optimizer); scaler.update()
            global_step+=1; running+=float(loss.item())*len(ids); seen+=len(ids)

            if global_step % 100 == 0:
                print({'seed':seed,'epoch':epoch+1,'step':global_step,'loss':round(running/max(seen,1),5)})
            if global_step % CONTRASTIVE_CHECKPOINT_EVERY == 0:
                ck=OUTPUT_ROOT/f'contrastive_resume_seed_{seed}.pt'
                torch.save({'model':model.state_dict(),'optimizer':optimizer.state_dict(),'scaler':scaler.state_dict(),
                            'epoch':epoch,'global_step':global_step},ck)
        history.append({'seed':seed,'epoch':epoch+1,'loss':running/max(seen,1),'seconds':time.perf_counter()-t0})

    out=CONTR_ROOT/f'seed_{seed}'; out.mkdir(parents=True,exist_ok=True)
    model.save_pretrained(out)
    (out/'contrastive_config.json').write_text(json.dumps({
        'seed':seed,'epochs':CONTRASTIVE_EPOCHS,'batch_size':CONTRASTIVE_BATCH_SIZE,
        'lr':CONTRASTIVE_LR,'temperature':CONTRASTIVE_TEMPERATURE,'max_length':MAX_LENGTH,
        'method':'unsupervised SimCSE-style dropout views','labels_used':False
    },indent=2),encoding='utf-8')
    ck=OUTPUT_ROOT/f'contrastive_resume_seed_{seed}.pt'
    if ck.exists(): ck.unlink()
    del model,optimizer,scaler,loader,ds
    gc.collect();
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print({'contrastive_seed':seed,'status':'COMPLETE','path':str(out)})
    return out, history

CONTRASTIVE_DIRS={}
contrast_hist=[]
for seed in SEEDS:
    p,h=train_contrastive(seed)
    CONTRASTIVE_DIRS[seed]=Path(p); contrast_hist.extend(h)

pd.DataFrame(contrast_hist).to_csv(OUTPUT_ROOT/'contrastive_training_history.csv',index=False)
print({k:str(v) for k,v in CONTRASTIVE_DIRS.items()})

## 05 – Repräsentationen extrahieren und cachen

Für **alle** Varianten wird der rohe `<s>`/CLS-Vektor (`last_hidden_state[:, 0, :]`) verwendet. Dadurch bleibt die Pooling-Regel konstant. Die Transformer-Encoder werden im Downstream-Schritt eingefroren; trainiert werden nur LR, XGBoost bzw. MLP auf den resultierenden Embeddings.

In [ ]:
# ============================================================
# 05 – Embedding-Extraktion
# ============================================================

EMBED_ROOT=OUTPUT_ROOT/'embeddings'
EMBED_ROOT.mkdir(exist_ok=True)


def model_dir_for(rep,seed):
    if rep=='BASE': return BASE_MODEL_DIR
    if rep=='DAPT': return DAPT_DIRS[seed]
    if rep=='CONTRASTIVE': return CONTRASTIVE_DIRS[seed]
    raise KeyError(rep)


def embedding_path(rep,seed,split):
    seed_key='shared' if rep=='BASE' else f'seed_{seed}'
    p=EMBED_ROOT/rep.lower()/seed_key
    p.mkdir(parents=True,exist_ok=True)
    return p/f'{split}.npy'

@torch.no_grad()
def extract_embeddings(rep,seed,split):
    path=embedding_path(rep,seed,split)
    if path.exists():
        arr=np.load(path,mmap_mode='r')
        if arr.shape[0]==len(TOKENS[split]['input_ids']):
            return path

    model=AutoModel.from_pretrained(model_dir_for(rep,seed),local_files_only=True)
    model.to(DEVICE); model.eval()
    hidden=int(model.config.hidden_size)
    n=len(TOKENS[split]['input_ids'])
    out=np.lib.format.open_memmap(path,mode='w+',dtype=np.float32,shape=(n,hidden))
    ds=TokenArrayDataset(TOKENS[split])
    loader=DataLoader(ds,batch_size=EMBED_BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,
                      pin_memory=torch.cuda.is_available())
    pos=0
    for ids,mask in loader:
        ids=ids.to(DEVICE,non_blocking=True); mask=mask.to(DEVICE,non_blocking=True)
        with torch.amp.autocast('cuda',enabled=torch.cuda.is_available()):
            z=model(input_ids=ids,attention_mask=mask).last_hidden_state[:,0,:]
        z=z.float().cpu().numpy()
        out[pos:pos+len(z)]=z; pos+=len(z)
    out.flush(); del out,model,loader,ds
    gc.collect();
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print({'embedding':rep,'seed':seed,'split':split,'n':n})
    return path

if RUN_EMBEDDINGS:
    # BASE ist deterministisch und wird genau einmal gespeichert.
    for split in ['train','calibration','iid','holdout']:
        extract_embeddings('BASE',SEEDS[0],split)
    for rep in ['DAPT','CONTRASTIVE']:
        for seed in SEEDS:
            for split in ['train','calibration','iid','holdout']:
                extract_embeddings(rep,seed,split)

print('Embedding-Cache vollständig.')

## 06 – Verschachtelte Labelbudgets

Pro Seed wird zunächst eine zufällige, klassenstratifizierte Reihenfolge festgelegt. 10 % ist Teilmenge von 25 %, 25 % ist Teilmenge von 100 %. Dieselben Indizes werden für BASE, DAPT und CONTRASTIVE sowie alle drei Downstream-Klassifikatoren verwendet.

In [ ]:
# ============================================================
# 06 – Nested Label Budgets
# ============================================================

y_train=train_df['label'].to_numpy(dtype=int)


def nested_budget_indices(y,seed):
    rng=np.random.default_rng(seed)
    class_orders={}
    for c in [0,1]:
        idx=np.flatnonzero(y==c).copy(); rng.shuffle(idx); class_orders[c]=idx
    result={}
    for frac in LABEL_BUDGETS:
        parts=[]
        for c in [0,1]:
            n=len(class_orders[c]) if frac==1.0 else max(1,int(round(len(class_orders[c])*frac)))
            parts.append(class_orders[c][:n])
        result[frac]=np.sort(np.concatenate(parts)).astype(np.int32)
    assert set(result[0.10]).issubset(set(result[0.25]))
    assert set(result[0.25]).issubset(set(result[1.00]))
    return result

BUDGET_INDICES={seed:nested_budget_indices(y_train,seed) for seed in SEEDS}
rows=[]
for seed,d in BUDGET_INDICES.items():
    for frac,idx in d.items():
        rows.append({'seed':seed,'budget':frac,'n':len(idx),'benign':int((y_train[idx]==0).sum()),'phish':int((y_train[idx]==1).sum())})
pd.DataFrame(rows).to_csv(OUTPUT_ROOT/'label_budget_audit.csv',index=False)
print(pd.DataFrame(rows).to_string(index=False))

## 07 – Downstream-Tuning

Um weder einen Strawman zu erzeugen noch für jede Repräsentation völlig andere Modellfamilien zu verwenden, wird **pro Klassifikatortyp genau ein Parametersatz** gewählt. Die Kandidaten werden auf dem 100-%-Trainingssplit per 3-fold-CV bewertet; der Score wird über BASE, DAPT(seed 42) und CONTRASTIVE(seed 42) gemittelt. Danach bleiben die Parameter für sämtliche Seeds, Labelbudgets und Testszenarien konstant.

In [ ]:
# ============================================================
# 07 – Gemeinsames, train-only Downstream-Tuning
# ============================================================

CANDIDATES={
    'LOGREG':[
        {'C':0.1},{'C':1.0},{'C':10.0}
    ],
    'XGBOOST':[
        {'n_estimators':300,'max_depth':3,'learning_rate':0.05,'min_child_weight':1},
        {'n_estimators':500,'max_depth':3,'learning_rate':0.05,'min_child_weight':1},
        {'n_estimators':400,'max_depth':5,'learning_rate':0.05,'min_child_weight':1},
        {'n_estimators':400,'max_depth':4,'learning_rate':0.10,'min_child_weight':2},
    ],
    'MLP':[
        {'hidden_layer_sizes':(128,), 'alpha':1e-4},
        {'hidden_layer_sizes':(128,), 'alpha':1e-3},
        {'hidden_layer_sizes':(256,64), 'alpha':1e-4},
        {'hidden_layer_sizes':(256,64), 'alpha':1e-3},
    ],
}


def build_classifier(kind,params,seed):
    if kind=='LOGREG':
        return Pipeline([
            ('scale',StandardScaler()),
            ('clf',LogisticRegression(C=params['C'],max_iter=2500,solver='lbfgs',random_state=seed))
        ])
    if kind=='XGBOOST':
        return XGBClassifier(
            **params, subsample=0.9,colsample_bytree=0.8,reg_lambda=1.0,
            objective='binary:logistic',eval_metric='logloss',tree_method='hist',
            n_jobs=2,random_state=seed
        )
    if kind=='MLP':
        return Pipeline([
            ('scale',StandardScaler()),
            ('clf',MLPClassifier(
                **params,activation='relu',solver='adam',batch_size=128,
                learning_rate_init=1e-3,max_iter=200,early_stopping=True,
                validation_fraction=0.15,n_iter_no_change=12,random_state=seed
            ))
        ])
    raise KeyError(kind)


def load_emb(rep,seed,split):
    s=SEEDS[0] if rep=='BASE' else seed
    return np.asarray(np.load(embedding_path(rep,s,split),mmap_mode='r'),dtype=np.float32)

# Repräsentationsauswahl für die gemeinsame Tuningentscheidung.
TUNE_REPS={rep:load_emb(rep,42,'train') for rep in REPRESENTATIONS}
cv=StratifiedKFold(n_splits=CV_FOLDS,shuffle=True,random_state=TUNING_SEED)

tuning_rows=[]; BEST_PARAMS={}
for kind in CLASSIFIERS:
    best=None
    for ci,params in enumerate(CANDIDATES[kind]):
        rep_scores=[]
        for rep,X in TUNE_REPS.items():
            fold_scores=[]
            for fold,(tr,va) in enumerate(cv.split(X,y_train)):
                model=build_classifier(kind,params,TUNING_SEED+fold)
                model.fit(X[tr],y_train[tr])
                p=model.predict_proba(X[va])[:,1]
                fold_scores.append(average_precision_score(y_train[va],p))
            rep_scores.append(np.mean(fold_scores))
            tuning_rows.append({'classifier':kind,'candidate':ci,'params':json.dumps(params),
                                'representation':rep,'cv_ap':float(np.mean(fold_scores))})
        mean_score=float(np.mean(rep_scores))
        if best is None or mean_score>best[0]: best=(mean_score,params)
    BEST_PARAMS[kind]=best[1]
    print({'classifier':kind,'best_mean_cv_ap':best[0],'params':best[1]})

pd.DataFrame(tuning_rows).to_csv(OUTPUT_ROOT/'classifier_tuning.csv',index=False)
(OUTPUT_ROOT/'best_classifier_params.json').write_text(json.dumps(BEST_PARAMS,indent=2),encoding='utf-8')

del TUNE_REPS
gc.collect()

## 08 – Finale Downstream-Evaluation

Jede Modellkombination wird auf dem jeweiligen verschachtelten Trainingsbudget fitten. Der operative Schwellenwert wird **nur** auf dem separaten Kalibrierungsanteil des früheren Validation-Splits so gewählt, dass dort die Ziel-FPR von 0,5 % möglichst eingehalten wird. Danach wird derselbe Schwellenwert unverändert auf IID und alle Shift-Szenarien übertragen.

Damit bleiben insbesondere steigende FPR-Werte unter Shift sichtbar und werden nicht durch testset-spezifische Nachkalibrierung verdeckt.

In [ ]:
# ============================================================
# 08 – Finale Evaluation
# ============================================================

RESULTS_PATH=OUTPUT_ROOT/'results_long.csv'


def threshold_for_target_fpr(y_true,score,target_fpr):
    y=np.asarray(y_true); s=np.asarray(score)
    neg=np.sort(s[y==0])[::-1]
    if len(neg)==0: return 0.5
    allowed=int(math.floor(target_fpr*len(neg)+1e-12))
    if allowed<=0:
        return float(np.nextafter(neg[0],np.inf))
    if allowed>=len(neg):
        return float(-np.inf)
    # Genau die obersten `allowed` Negativen dürfen oberhalb der Schwelle liegen.
    return float(np.nextafter(neg[allowed],np.inf))


def metric_row(y,score,threshold):
    pred=(score>=threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {
        'average_precision':float(average_precision_score(y,score)),
        'roc_auc':float(roc_auc_score(y,score)) if len(np.unique(y))==2 else np.nan,
        'precision':float(precision_score(y,pred,zero_division=0)),
        'recall':float(recall_score(y,pred,zero_division=0)),
        'f1':float(f1_score(y,pred,zero_division=0)),
        'empirical_fpr':float(fp/max(fp+tn,1)),
        'tp':int(tp),'fp':int(fp),'tn':int(tn),'fn':int(fn),
    }


def get_eval_data(rep,seed,scenario):
    source,idx=SCENARIOS[scenario]
    X=load_emb(rep,seed,source)
    y=(iid_df if source=='iid' else holdout_df)['label'].to_numpy(dtype=int)
    return X[idx],y[idx]

existing=[]
if RESULTS_PATH.exists():
    existing=pd.read_csv(RESULTS_PATH).to_dict('records')
completed={(r['representation'],r['classifier'],int(r['seed']),float(r['label_budget']),r['scenario']) for r in existing}
results=list(existing)

if RUN_DOWNSTREAM:
    for seed in SEEDS:
        for frac in LABEL_BUDGETS:
            idx=BUDGET_INDICES[seed][frac]
            yb=y_train[idx]
            for rep in REPRESENTATIONS:
                Xtrain=load_emb(rep,seed,'train')[idx]
                Xcal=load_emb(rep,seed,'calibration')
                ycal=calibration_df['label'].to_numpy(dtype=int)
                for kind in CLASSIFIERS:
                    scenario_keys=[(rep,kind,seed,float(frac),s) for s in SCENARIOS]
                    if all(k in completed for k in scenario_keys):
                        continue
                    t0=time.perf_counter()
                    model=build_classifier(kind,BEST_PARAMS[kind],seed)
                    model.fit(Xtrain,yb)
                    fit_seconds=time.perf_counter()-t0

                    cal_score=model.predict_proba(Xcal)[:,1]
                    threshold=threshold_for_target_fpr(ycal,cal_score,TARGET_FPR)
                    cal_metrics=metric_row(ycal,cal_score,threshold)

                    for scenario in SCENARIOS:
                        key=(rep,kind,seed,float(frac),scenario)
                        if key in completed: continue
                        Xev,yev=get_eval_data(rep,seed,scenario)
                        t1=time.perf_counter(); score=model.predict_proba(Xev)[:,1]; pred_seconds=time.perf_counter()-t1
                        row={
                            'representation':rep,'classifier':kind,'seed':seed,
                            'label_budget':float(frac),'n_train':int(len(idx)),
                            'scenario':scenario,'n_test':int(len(yev)),
                            'threshold':threshold,'target_fpr':TARGET_FPR,
                            'calibration_fpr':cal_metrics['empirical_fpr'],
                            'calibration_recall':cal_metrics['recall'],
                            'fit_seconds':fit_seconds,'predict_seconds':pred_seconds,
                            **metric_row(yev,score,threshold)
                        }
                        results.append(row); completed.add(key)
                    pd.DataFrame(results).to_csv(RESULTS_PATH,index=False)
                    print({'done':(rep,kind,seed,frac),'rows':len(results),'threshold':round(threshold,6)})
                    del model,Xtrain,Xcal,cal_score
                    gc.collect()

results_df=pd.DataFrame(results)
print(results_df.shape)
print(results_df.head())

## 09 – Freeze-Auswertungen: ΔSSL und ΔShift

- **ΔSSL:** Differenz zwischen DAPT/CONTRASTIVE und BASE innerhalb desselben Klassifikators, Seeds, Labelbudgets und Szenarios.
- **ΔShift:** Differenz zwischen IID und einem Shift-Szenario innerhalb derselben Modellkonfiguration.

Positive `delta_ssl_*`-Werte bedeuten einen Vorteil der jeweiligen SSL-Repräsentation gegenüber BASE. Positive `drop_*`-Werte bedeuten Leistungsverlust vom IID- zum Shift-Szenario.

In [ ]:
# ============================================================
# 09 – Effekt- und Robustheitsmatrizen
# ============================================================

KEY=['classifier','seed','label_budget','scenario']
base=results_df[results_df.representation.eq('BASE')][KEY+['average_precision','recall','f1','empirical_fpr']].rename(columns={
    'average_precision':'base_ap','recall':'base_recall','f1':'base_f1','empirical_fpr':'base_fpr'
})
ssl=results_df[~results_df.representation.eq('BASE')].merge(base,on=KEY,how='left')
ssl['delta_ssl_ap']=ssl['average_precision']-ssl['base_ap']
ssl['delta_ssl_recall']=ssl['recall']-ssl['base_recall']
ssl['delta_ssl_f1']=ssl['f1']-ssl['base_f1']
ssl['delta_ssl_fpr']=ssl['empirical_fpr']-ssl['base_fpr']
ssl.to_csv(OUTPUT_ROOT/'ssl_effects.csv',index=False)

IID=results_df[results_df.scenario.eq('IID')][['representation','classifier','seed','label_budget','average_precision','recall','f1','empirical_fpr']].rename(columns={
    'average_precision':'iid_ap','recall':'iid_recall','f1':'iid_f1','empirical_fpr':'iid_fpr'
})
shift=results_df[~results_df.scenario.eq('IID')].merge(IID,on=['representation','classifier','seed','label_budget'],how='left')
shift['drop_ap']=shift['iid_ap']-shift['average_precision']
shift['drop_recall']=shift['iid_recall']-shift['recall']
shift['drop_f1']=shift['iid_f1']-shift['f1']
shift['fpr_increase']=shift['empirical_fpr']-shift['iid_fpr']
shift['ap_retention']=shift['average_precision']/shift['iid_ap'].replace(0,np.nan)
shift.to_csv(OUTPUT_ROOT/'shift_degradation.csv',index=False)

summary=(results_df.groupby(['representation','classifier','label_budget','scenario'])
         .agg(ap_mean=('average_precision','mean'),ap_sd=('average_precision','std'),
              recall_mean=('recall','mean'),recall_sd=('recall','std'),
              f1_mean=('f1','mean'),f1_sd=('f1','std'),
              fpr_mean=('empirical_fpr','mean'),fpr_sd=('empirical_fpr','std'))
         .reset_index())
summary.to_csv(OUTPUT_ROOT/'summary_metrics.csv',index=False)

print(summary.head(20).to_string(index=False))

In [ ]:
# ============================================================
# 10 – Gepaarte 95%-Konfidenzintervalle für ΔSSL und ΔShift
# ============================================================

def mean_ci(values,alpha=0.05):
    x=np.asarray(values,dtype=float); x=x[np.isfinite(x)]
    if len(x)<2: return (float(np.mean(x)) if len(x) else np.nan,np.nan,np.nan)
    m=float(np.mean(x)); se=stats.sem(x); q=stats.t.ppf(1-alpha/2,len(x)-1)
    return m,float(m-q*se),float(m+q*se)

ci_rows=[]
for keys,g in ssl.groupby(['representation','classifier','label_budget','scenario']):
    for metric in ['delta_ssl_ap','delta_ssl_recall','delta_ssl_f1']:
        m,lo,hi=mean_ci(g[metric])
        ci_rows.append({
            'effect':'SSL_vs_BASE','representation':keys[0],'classifier':keys[1],
            'label_budget':keys[2],'scenario':keys[3],'metric':metric,
            'mean':m,'ci95_low':lo,'ci95_high':hi,'n_seeds':len(g)
        })
for keys,g in shift.groupby(['representation','classifier','label_budget','scenario']):
    for metric in ['drop_ap','drop_recall','drop_f1']:
        m,lo,hi=mean_ci(g[metric])
        ci_rows.append({
            'effect':'IID_to_SHIFT','representation':keys[0],'classifier':keys[1],
            'label_budget':keys[2],'scenario':keys[3],'metric':metric,
            'mean':m,'ci95_low':lo,'ci95_high':hi,'n_seeds':len(g)
        })

ci_df=pd.DataFrame(ci_rows)
ci_df.to_csv(OUTPUT_ROOT/'effect_confidence_intervals.csv',index=False)
print(ci_df.head(20).to_string(index=False))

## 11 – Ergebnisgrafiken

Die Grafiken werden ausschließlich aus `results_long.csv` bzw. den abgeleiteten Effektdateien erzeugt. Damit bleibt `results_long.csv` die Single Source of Truth.

In [ ]:
# ============================================================
# 11 – Finale Abbildungen
# ============================================================

import matplotlib.pyplot as plt
FIG_ROOT=OUTPUT_ROOT/'figures'; FIG_ROOT.mkdir(exist_ok=True)

if RUN_PLOTS:
    # A: ΔSSL AP nach Downstream und Szenario bei 100 % Labels
    plot=(ssl[ssl.label_budget.eq(1.0)]
          .groupby(['representation','classifier','scenario'])['delta_ssl_ap'].mean().reset_index())
    for scenario in plot['scenario'].unique():
        p=plot[plot.scenario.eq(scenario)]
        pivot=p.pivot(index='classifier',columns='representation',values='delta_ssl_ap').reindex(CLASSIFIERS)
        ax=pivot.plot(kind='bar',figsize=(9,5))
        ax.axhline(0,linewidth=1)
        ax.set_title(f'Delta SSL Average Precision – {scenario}')
        ax.set_ylabel('Δ Average Precision gegenüber BASE')
        ax.set_xlabel('Downstream-Klassifikator')
        plt.tight_layout(); plt.savefig(FIG_ROOT/f'delta_ssl_ap_{scenario.lower()}.png',dpi=220); plt.close()

    # B: AP-Retention unter Shift, 100 % Labels
    p=(shift[shift.label_budget.eq(1.0)]
       .groupby(['representation','classifier','scenario'])['ap_retention'].mean().reset_index())
    for clf in CLASSIFIERS:
        q=p[p.classifier.eq(clf)].pivot(index='scenario',columns='representation',values='ap_retention')
        ax=q.plot(kind='bar',figsize=(9,5))
        ax.set_title(f'AP-Retention unter Distribution Shift – {clf}')
        ax.set_ylabel('AP Shift / AP IID')
        ax.set_xlabel('Shift-Szenario')
        plt.tight_layout(); plt.savefig(FIG_ROOT/f'ap_retention_{clf.lower()}.png',dpi=220); plt.close()

    # C: Lernkurven pro Downstream für IID und kombiniertes OOD
    for clf in CLASSIFIERS:
        for scenario in ['IID','DOMAIN_TEMPLATE_OOD']:
            p=(results_df[(results_df.classifier.eq(clf)) & (results_df.scenario.eq(scenario))]
               .groupby(['representation','label_budget'])['average_precision'].mean().reset_index())
            fig,ax=plt.subplots(figsize=(8,5))
            for rep in REPRESENTATIONS:
                q=p[p.representation.eq(rep)].sort_values('label_budget')
                ax.plot(q['label_budget']*100,q['average_precision'],marker='o',label=rep)
            ax.set_xlabel('Gelabelte Trainingsdaten [%]'); ax.set_ylabel('Average Precision')
            ax.set_title(f'Labelbudget – {clf} – {scenario}'); ax.legend()
            plt.tight_layout(); plt.savefig(FIG_ROOT/f'label_curve_{clf.lower()}_{scenario.lower()}.png',dpi=220); plt.close()

print('Abbildungen:',len(list(FIG_ROOT.glob('*.png'))))

## 12 – Abschluss-Audit und Export

Der Abschlussblock prüft, ob alle eingefrorenen Kombinationen vorhanden sind. Erwartet werden:

`3 Repräsentationen × 3 Klassifikatoren × 5 Seeds × 3 Labelbudgets × 5 Szenarien = 675 Ergebniszeilen`.

Active Learning ist im Kernlauf deaktiviert.

In [ ]:
# ============================================================
# 12 – Completion Audit + ZIP
# ============================================================

EXPECTED_ROWS=len(REPRESENTATIONS)*len(CLASSIFIERS)*len(SEEDS)*len(LABEL_BUDGETS)*len(SCENARIOS)
unique_rows=results_df.drop_duplicates(['representation','classifier','seed','label_budget','scenario'])
if len(unique_rows)!=EXPECTED_ROWS:
    raise RuntimeError(f'Freeze unvollständig: {len(unique_rows)} / {EXPECTED_ROWS} eindeutige Ergebniszeilen.')

config={
    'freeze_version':'2026-08-08-final',
    'representations':REPRESENTATIONS,'classifiers':CLASSIFIERS,'seeds':SEEDS,
    'label_budgets':LABEL_BUDGETS,'scenarios':list(SCENARIOS.keys()),
    'target_fpr':TARGET_FPR,'max_length':MAX_LENGTH,
    'calibration_fraction':CALIBRATION_FRACTION,
    'contrastive':{
        'method':'unsupervised SimCSE-style dropout views','epochs':CONTRASTIVE_EPOCHS,
        'batch_size':CONTRASTIVE_BATCH_SIZE,'lr':CONTRASTIVE_LR,
        'temperature':CONTRASTIVE_TEMPERATURE,'labels_used':False,
    },
    'best_classifier_params':BEST_PARAMS,
    'active_learning_core':False,
}
(OUTPUT_ROOT/'configuration.json').write_text(json.dumps(config,indent=2),encoding='utf-8')

complete={
    'status':'COMPLETE','expected_rows':EXPECTED_ROWS,'actual_rows':len(unique_rows),
    'completed_utc':pd.Timestamp.utcnow().isoformat(),
    'primary_metric':'average_precision',
    'operational_target_fpr':TARGET_FPR,
}
(OUTPUT_ROOT/'FINAL_FREEZE_COMPLETE.json').write_text(json.dumps(complete,indent=2),encoding='utf-8')

zip_base='/kaggle/working/phreshphish_final_freeze_results'
shutil.make_archive(zip_base,'zip',OUTPUT_ROOT)
print(complete)
print('ZIP:',zip_base+'.zip')

## Interpretation der erzeugten Dateien

- `results_long.csv`: primäre Ergebnistabelle aller 675 Kombinationen
- `summary_metrics.csv`: Mittelwerte und Standardabweichungen über fünf Seeds
- `ssl_effects.csv`: DAPT/Contrastive minus BASE innerhalb identischer Downstream-Bedingungen
- `shift_degradation.csv`: IID minus Shift sowie AP-Retention
- `effect_confidence_intervals.csv`: gepaarte 95-%-Konfidenzintervalle über die Seeds
- `classifier_tuning.csv`: ausschließlich train-basierte Hyperparameterwahl
- `label_budget_audit.csv`: Nachweis der verschachtelten Labelbudgets
- `scenario_summary.csv`: Größe und Klassenverteilung der Testszenarien
- `artifact_audit.json`: tatsächlich verwendete Eingabeartefakte
- `contrastive_training_history.csv`: Contrastive-Trainingsverlauf
- `figures/`: automatisch erzeugte Ergebnisabbildungen
- `FINAL_FREEZE_COMPLETE.json`: Completion-Marker
- `/kaggle/working/phreshphish_final_freeze_results.zip`: gebündelter Ergebnisexport